# Macroeconomic Analysis Studio with LLM prompt writing
Write the economic question below, then run the notebook. A local Llama prompt reviewer checks whether it has enough information. The analyst then writes Python, a critic reviews what it does, and both approve the same revision. The numerical result and its interpretation are reviewed before release.

For the browser version, use `econ422-studio app`. This notebook uses the same live workflow. Install the package in this kernel's environment, start Ollama, and download `llama3.1:8b` once. The CSV is included. The included series is historical U.S. retail trade sales in millions of dollars.

In [ ]:
from studio_core.api import PromptNeedsAnswer, PromptRewriteOffered, analyze
from studio_core.data_io import load_retail_demo
from studio_core.example_prompt import RETAIL_PROMPTS, EXPLORE_PROMPTS, RETAIL_SOURCE

data = load_retail_demo()
print(data.head().to_string(index=False))

## 1. Write a prompt
Use the five-field forecast prompt or switch to EXPLORE_PROMPTS[0] to measure features and choose a forecasting model. You can edit your wording before running.

In [ ]:
prompt = RETAIL_PROMPTS[0]
print(prompt)

## 2. Follow the generated Python and the exchange
The callback prints each message as it arrives. A local model can accept the first proposal or request revisions. If the prompt reviewer asks for a detail, enter your answer at the input below to continue with the same data. For rough writing, review the proposed rewrite and choose whether to use it.

In [ ]:
def show_event(event):
    label = event.get("decision", "")
    stage = f"Round {event['round']}" if event["round"] else "Prompt review"
    print(f"{stage} | {event['role']} {label}", flush=True)
    print(event["message"], flush=True)
    if event.get("suggested_prompt") and event.get("decision") == "ask_user":
        print("Suggested wording:", event["suggested_prompt"], flush=True)
    if "code" in event:
        print(event["code"], flush=True)

clarifications = []
wording = None
while True:
    try:
        result = analyze(prompt, data=data, data_source=RETAIL_SOURCE, on_event=show_event, clarifications=clarifications,
                         pause_for_rewrite=True, wording=wording)
        break
    except PromptRewriteOffered as offer:
        print("Suggested wording:", offer)
        reply = input("Type y to use it, type your own rewrite, or press Enter to keep the original: ").strip()
        action = "keep" if not reply else "use" if reply.lower() == "y" else "edit"
        wording = {"action": action, "suggestion": str(offer), "edited": reply if action == "edit" else ""}
    except PromptNeedsAnswer as question:
        if len(clarifications) == 6:
            raise
        answer = input(f"{question} Your answer: ").strip()
        if not answer:
            raise
        clarifications.append({"question": str(question), "answer": answer})

## 3. Read the approved result
The forecast table compares models on the last 24 months. The exploration report uses earlier months to measure features, compare candidates and recommend a model.

In [ ]:
print(result.audit["interpretation"])
print(result.metrics.round(4).to_string())
if result.audit["task"] == "forecast":
    print("Weights:", result.audit["weights"])
    print("Next forecast:", result.audit["next_month"])
elif result.audit["task"] == "exploration":
    print(result.audit["verdict"])

## 4. Inspect or replay the accepted Python
The script can run independently of Llama. With custom data, supply the same CSV path when replaying it.

In [ ]:
print(result.audit["python_script"])
# Optional: choose a new directory each time.
# result.save("accepted_analysis")